In [6]:
from numpy.ma.core import masked_values
%env TF_CPP_MIN_LOG_LEVEL=2

env: TF_CPP_MIN_LOG_LEVEL=2


In [ ]:
import tensorflow as tf

print("CUDA Built:", tf.test.is_built_with_cuda())
print("Physical GPUs:", tf.config.list_physical_devices('GPU'))

CUDA Built: True
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
import numpy as np
import tensorflow_datasets as tfds

train_ds,test_ds,validation_ds = tfds.load("gigaword", split=['train', 'test', 'validation'], as_supervised=True)

#Because the dataset is too large, we will slice it for training and testing. We are using 80/20 split for training and testing respectively.
train_ds = train_ds.take(100000)
test_ds = test_ds.take(2000)
validation_ds = validation_ds.take(2000)

#Coverting the dataset into numpy arrays for training and testing.
X_train = np.array([x.numpy() for x, y in train_ds])
y_train = np.array([y.numpy() for x, y in train_ds])

X_test = np.array([x.numpy() for x, y in test_ds])
y_test = np.array([y.numpy() for x, y in test_ds])

X_validation = np.array([x.numpy() for x, y in validation_ds])
y_validation = np.array([y.numpy() for x, y in validation_ds])

#To know the shape of the dataset.
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape, X_validation.shape, y_validation.shape)

(100000,) (100000,) (1951,) (1951,) (2000,) (2000,)


In [4]:
print(y_train[0], X_train[0], sep="\n")

b'fair trade commission investigating consumer price hike'
b'officials of the cabinet-level fair trade commission -lrb- ftc -rrb- said friday that they have formed an ad hoc group to investigate whether there is any manipulation of commodity prices by traders in local market .'


In [ ]:
# The data have the b"" prefix, we will remove it.
X_train, X_test, X_validation, y_train, y_test, y_validation = [
    np.array([x.decode('utf-8') for x in split])
    for split in (X_train, X_test, X_validation, y_train, y_test, y_validation)
]

# let see the data again.
print(y_train[0], X_train[0], sep="\n")

fair trade commission investigating consumer price hike
officials of the cabinet-level fair trade commission -lrb- ftc -rrb- said friday that they have formed an ad hoc group to investigate whether there is any manipulation of commodity prices by traders in local market .


The data is now ready for preprocessing. We will use the Tokenizer class from Keras to convert the text data into sequences of integers. This will allow us to feed the data into a neural network model for training.

Tokenization easily make the vocabulary to big, it makes the model easily Out of vocabulary (OOV). So we apply subword tokenization to reduce the vocabulary size.

In [13]:
from keras_hub.tokenizers import compute_word_piece_vocabulary
from tensorflow.data import Dataset

def WordPiece(texts,NUM_WORDS):
    vocab = compute_word_piece_vocabulary(Dataset.from_tensor_slices(texts),
                                          vocabulary_size=NUM_WORDS,
                                          lowercase=True)
    return vocab

In [14]:
def pad_targets(split, tokenizer, start_id, end_id, maxlen): #Python list is slow so we use numpy array.
    tokens = tokenizer.tokenize(tf.convert_to_tensor(split))
    pieces = sequence.pad_sequences(
        tokens, maxlen=maxlen - 2, padding="post", truncating="post"
    )

    result = np.zeros((len(pieces), maxlen), dtype="int32")
    result[:, 0] = start_id
    result[:, 1:-1] = pieces
    lengths = np.count_nonzero(pieces, axis=1)
    result[np.arange(len(pieces)), lengths + 1] = end_id
    return result

In [ ]:
X_train_pad, y_train_pad, X_test_pad, y_test_pad, X_validation_pad, y_validation_pad, tokenizer_y, tokenizer_x = preprocess_data(X_train, y_train, X_test, y_test, X_validation, y_validation) #Save the preprocessed date. We Don't want to wait for 4 mins for it.
np.save("wordpiece_vocab_x.npy", np.asarray(tokenizer_x.get_vocabulary(), dtype=str))
np.save("wordpiece_vocab_y.npy", np.asarray(tokenizer_y.get_vocabulary(), dtype=str))
np.savez("tokenized_data.npz",
         X_train=X_train_pad,
         y_train=y_train_pad,
         X_validation=X_validation_pad,
         y_validation=y_validation_pad,
         X_test=X_test_pad,
         y_test=y_test_pad,
)

KeyboardInterrupt: 

In [15]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import sequence
from keras_hub.tokenizers import WordPieceTokenizer

NUM_WORDS = 10000       # Number of most frequent words to use
INPUT_SEQ_LEN = 100      # Maximum article length
TARGET_SEQ_LEN = 50     # Maximum summary length

def preprocess_data(X_train, y_train, X_test, y_test, X_validation, y_validation):

    vocab_y = WordPiece(y_train, NUM_WORDS)
    vocab_x = WordPiece(X_train, NUM_WORDS)

    tokenizer_x = WordPieceTokenizer(vocabulary=vocab_x, lowercase=True)

    X_train_pad, X_test_pad, X_validation_pad = [
        sequence.pad_sequences(
            tokenizer_x.tokenize(tf.convert_to_tensor(split)),
            maxlen=INPUT_SEQ_LEN,
            padding="post",
            truncating="post",
        )
        for split in (X_train, X_test, X_validation)
    ]

    # Start and end tokens
    tokenizer_y = WordPieceTokenizer(vocabulary=vocab_y, lowercase=True)
    start_id = vocab_y.index("[CLS]")
    end_id = vocab_y.index("[SEP]")

    y_train_pad, y_test_pad, y_validation_pad = [
    pad_targets(split, tokenizer_y, start_id, end_id, TARGET_SEQ_LEN)
    for split in (y_train, y_test, y_validation)
]

    return X_train_pad, y_train_pad, X_test_pad, y_test_pad, X_validation_pad, y_validation_pad, tokenizer_y, tokenizer_x

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

early_stopping = EarlyStopping(monitor="val_loss",
                               patience=5,
                               restore_best_weights=True)

checkpoint = ModelCheckpoint("Summarize.keras",
                             monitor="val_loss",
                             save_best_only=True)

def compile_and_fit_model(model, X_train, y_train, X_valid, y_valid,
                          epochs, batch_size, verbose):

    valid_mask = (y_valid != 0).astype("float32")
    train_mask = (y_train != 0).astype("float32")

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam",
        weighted_metrics=["acc"]
    )

    history = model.fit(
        X_train,
        y_train,
        sample_weight=train_mask,
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
        validation_data=(X_valid, y_valid, valid_mask),
        callbacks=[early_stopping, checkpoint]
    )

    return history

In [ ]:
def history_final_values(history):
    return {k: round(v[-1], 4) for k, v in history.history.items()}

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_history(history):
    df = pd.DataFrame(history.history)
    df.index = df.index + 1
    fig, axs = plt.subplots(1, 2, figsize=(8, 4))
    df.plot(y=["loss", "val_loss"], grid=True, ax=axs[0])
    df.plot(y=["acc", "val_acc"], grid=True, ax=axs[1])
    plt.tight_layout()

In [ ]:
final_values = {}

The model uses a sequence-to-sequence encoder–decoder architecture. The encoder processes the article and produces fixed-size hidden and cell states, which initialize the decoder. This structure supports input and output sequences of different lengths, making it suitable for text summarization. However, the fixed-size states may limit how much information reaches the decoder.

The model structure is:

- **Encoder:**
  - Input layer for article tokens.
  - Embedding layer to convert tokens into dense vectors.
  - Convolutional layer to extract local features.
  - Bidirectional LSTM to process the sequence and produce hidden and cell states.
- **Decoder:**
  - Input layer for summary tokens.
  - Embedding layer to convert tokens into dense vectors.
  - LSTM initialized with the encoder states.
  - Dropout layer to reduce the risk of overfitting.
  - TimeDistributed dense layer with softmax to predict word probabilities at each time step.
- **Loss function:** Sparse categorical cross-entropy, suitable for integer token labels.
- **Optimizer:** Adam, which adapts the learning rate during training.
- **Metric:** Accuracy over non-padding target tokens.


During training, we use **teacher forcing**: the decoder receives the correct previous headline tokens. During independent generation, it instead uses its own previous predictions. Teacher forcing helps the decoder learn from correct context, but does not guarantee accurate generated summaries.

In [ ]:
# Model training
from keras import ops

#preprocessing the data
vocab_x = np.load("wordpiece_vocab_x.npy", allow_pickle=False).tolist()
vocab_y = np.load("wordpiece_vocab_y.npy", allow_pickle=False).tolist()

tokenizer_x = WordPieceTokenizer(vocabulary=vocab_x, lowercase=True)
tokenizer_y = WordPieceTokenizer(vocabulary=vocab_y, lowercase=True)
with np.load("tokenized_data.npz", allow_pickle=False) as data:
    X_train_pad = data["X_train"]
    y_train_pad = data["y_train"]
    X_validation_pad = data["X_validation"]
    y_validation_pad = data["y_validation"]
    X_test_pad = data["X_test"]
    y_test_pad = data["y_test"]

#Shifting the target sequence by one time step to the right, so that the decoder can learn to predict the next token in the sequence given the previous tokens.
decoder_input_train = y_train_pad[:, :-1]
decoder_target_train = y_train_pad[:, 1:]

decoder_input_val = y_validation_pad[:, :-1]
decoder_target_val = y_validation_pad[:, 1:]

#ENCODER
encoder_input = keras.Input(shape=(INPUT_SEQ_LEN,),dtype="int32") #An input for the article tokens.
encoder_mask = ops.not_equal(encoder_input, 0) #A mask to ignore padding tokens in the input sequence.

encoder_embedding = layers.Embedding(len(vocab_x), 64)(encoder_input)#load the data
mask_values = ops.cast(ops.expand_dims(encoder_mask, axis=-1), dtype=encoder_embedding.dtype)
encoder_embedding = encoder_embedding * mask_values #Apply the mask to the embedded article tokens.

encoder_cnn = layers.Conv1D(64, 5, padding="same", activation="relu")(encoder_embedding) #A convolutional layer to extract features from the embedded article tokens.

sequence_layer  = layers.Bidirectional(layers.LSTM(64,return_state=True,return_sequences=True))
result = sequence_layer(encoder_cnn, mask=encoder_mask) #A bidirectional LSTM layer to capture the sequential dependencies in the article tokens.

encoder_output, forward_h, forward_c, backward_h, backward_c = result

#join the forward and backward hidden states
state_h = layers.Concatenate()([forward_h, backward_h]) # combine both hidden and cell states
state_c = layers.Concatenate()([forward_c, backward_c])
encoder_states = [state_h, state_c] # make it a list to pass to the decoder

#DECODER
decoder_input = keras.Input(shape=(TARGET_SEQ_LEN-1,),dtype="int32")
decoder_embedding = layers.Embedding(len(vocab_y), 64)(decoder_input)
decoder_lstm = layers.LSTM(128, return_sequences=True, return_state=True)
decoder_output, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

attention = layers.Attention()
context = attention([decoder_output, encoder_output],mask=[None, encoder_mask])

decoder_output = layers.Concatenate()([decoder_output, context])
decoder_output = layers.Dropout(0.2)(decoder_output)
decoder_dense = layers.TimeDistributed(layers.Dense(len(vocab_y), activation="softmax"))
decoder_output = decoder_dense(decoder_output) #predicts the word probabilities for each time step in the output sequence.

model = keras.Model([encoder_input, decoder_input],
                    decoder_output,
                    name="Seq2Seq_CNN_LSTM")

# Model training
history = compile_and_fit_model(
    model,
    [X_train_pad, decoder_input_train],
    decoder_target_train,
    [X_validation_pad, decoder_input_val],
    decoder_target_val,
    epochs=100,
    batch_size=256,
    verbose=2
)
final_values[model.name] = history_final_values(history)
print(final_values[model.name])


Epoch 1/100
391/391 - 74s - 189ms/step - acc: 0.1244 - loss: 1.5609 - val_acc: 0.1492 - val_loss: 1.4901
Epoch 2/100
391/391 - 69s - 177ms/step - acc: 0.1626 - loss: 1.4128 - val_acc: 0.1701 - val_loss: 1.3890
Epoch 3/100
391/391 - 69s - 177ms/step - acc: 0.1800 - loss: 1.3307 - val_acc: 0.1885 - val_loss: 1.3218
Epoch 4/100
391/391 - 69s - 177ms/step - acc: 0.1979 - loss: 1.2687 - val_acc: 0.2079 - val_loss: 1.2673
Epoch 5/100
391/391 - 61s - 157ms/step - acc: 0.2149 - loss: 1.2173 - val_acc: 0.2212 - val_loss: 1.2274
Epoch 6/100
391/391 - 54s - 138ms/step - acc: 0.2287 - loss: 1.1758 - val_acc: 0.2354 - val_loss: 1.1947
Epoch 7/100
391/391 - 54s - 138ms/step - acc: 0.2418 - loss: 1.1403 - val_acc: 0.2459 - val_loss: 1.1677
Epoch 8/100
391/391 - 62s - 159ms/step - acc: 0.2530 - loss: 1.1089 - val_acc: 0.2552 - val_loss: 1.1445
Epoch 9/100
391/391 - 70s - 178ms/step - acc: 0.2639 - loss: 1.0807 - val_acc: 0.2624 - val_loss: 1.1257
Epoch 10/100
391/391 - 70s - 178ms/step - acc: 0.2733 -

In [ ]:
history = compile_and_fit_model( #just in case the model is not trained enough, we can continue training it.
    model,
    [X_train_pad, decoder_input_train],
    decoder_target_train,
    [X_validation_pad, decoder_input_val],
    decoder_target_val,
    epochs=10,
    batch_size=256,
    verbose=2
)
final_values[model.name] = history_final_values(history)
print(final_values[model.name])


Epoch 1/20
391/391 - 76s - 194ms/step - acc: 0.2466 - loss: 0.9065 - val_acc: 0.2478 - val_loss: 0.9334
Epoch 2/20
391/391 - 74s - 190ms/step - acc: 0.2527 - loss: 0.8881 - val_acc: 0.2530 - val_loss: 0.9233
Epoch 3/20
391/391 - 75s - 191ms/step - acc: 0.2583 - loss: 0.8731 - val_acc: 0.2562 - val_loss: 0.9142
Epoch 4/20
391/391 - 75s - 192ms/step - acc: 0.2634 - loss: 0.8592 - val_acc: 0.2595 - val_loss: 0.9064
Epoch 5/20
391/391 - 74s - 190ms/step - acc: 0.2686 - loss: 0.8463 - val_acc: 0.2621 - val_loss: 0.9004
Epoch 6/20
391/391 - 75s - 192ms/step - acc: 0.2731 - loss: 0.8345 - val_acc: 0.2657 - val_loss: 0.8941
Epoch 7/20
391/391 - 76s - 194ms/step - acc: 0.2772 - loss: 0.8231 - val_acc: 0.2680 - val_loss: 0.8895
Epoch 8/20
391/391 - 75s - 192ms/step - acc: 0.2816 - loss: 0.8125 - val_acc: 0.2685 - val_loss: 0.8854
Epoch 9/20
391/391 - 76s - 194ms/step - acc: 0.2857 - loss: 0.8024 - val_acc: 0.2715 - val_loss: 0.8828
Epoch 10/20
391/391 - 77s - 197ms/step - acc: 0.2897 - loss: 0.7

In [ ]:
model.save("Summarize_attention.keras")

In [ ]:
model.summary()

Model: "Seq2Seq_CNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 100)       │          0 │ input_layer_8[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_4       │ (None, 100, 1)    │          0 │ not_equal_4[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 100, 64)   │    617,088 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast_4 (Cast)       │ (None, 100, 1)    │          0 │ expand_dims_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_4          │ (None, 100, 64)   │          0 │ embedding_10[0][… │
│ (Multiply)          │                   │            │ cast_4[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 100, 64)   │     20,544 │ multiply_4[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ [(None, 100,      │     66,048 │ conv1d_4[0][0],   │
│ (Bidirectional)     │ 128), (None, 64), │            │ not_equal_4[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_11        │ (None, 49, 64)    │    576,512 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_8       │ (None, 128)       │          0 │ bidirectional_4[… │
│ (Concatenate)       │                   │            │ bidirectional_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_9       │ (None, 128)       │          0 │ bidirectional_4[… │
│ (Concatenate)       │                   │            │ bidirectional_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ [(None, 49, 128), │     98,816 │ embedding_11[0][… │
│                     │ (None, 128),      │            │ concatenate_8[0]… │
│                     │ (None, 128)]      │            │ concatenate_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 49, 128)   │          0 │ lstm_9[0][0],     │
│ (Attention)         │                   │            │ bidirectional_4[… │
│                     │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_10      │ (None, 49, 256)   │          0 │ lstm_9[0][0],     │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 49, 256)   │          0 │ concatenate_10[0

 Total params: 11,082,194 (42.28 MB)

 Trainable params: 3,694,064 (14.09 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,388,130 (28.18 MB)

In [ ]:
plot_history(history)

AttributeError: 'Functional' object has no attribute 'history'

In [2]:
import keras
model = keras.models.load_model("Summarize_attention.keras")

In [3]:
from keras_hub.tokenizers import WordPieceTokenizer
import numpy as np
vocab_x = np.load("wordpiece_vocab_x.npy", allow_pickle=False).tolist()
vocab_y = np.load("wordpiece_vocab_y.npy", allow_pickle=False).tolist()

tokenizer_x = WordPieceTokenizer(vocabulary=vocab_x, lowercase=True)
tokenizer_y = WordPieceTokenizer(vocabulary=vocab_y, lowercase=True)
with np.load("tokenized_data.npz", allow_pickle=False) as data:
    X_train_pad = data["X_train"]
    y_train_pad = data["y_train"]
    X_validation_pad = data["X_validation"]
    y_validation_pad = data["y_validation"]
    X_test_pad = data["X_test"]
    y_test_pad = data["y_test"]

In [20]:
from tqdm import tqdm
import tensorflow as tf

def indenpendent_generate_summary(article_num, dataset=X_validation_pad, batch_size=8):
    if isinstance(article_num, (int, np.integer)):
        articles = dataset[article_num : article_num + 1]
    elif article_num.ndim == 1:
        articles = np.expand_dims(article_num, axis=0)
    else:
        articles = article_num

    end_id = vocab_y.index("[SEP]")
    pad_id = vocab_y.index("[PAD]")
    summaries = []

    for start in tqdm(range(0, len(articles), batch_size), desc="Generating batches"):
        article = articles[start : start + batch_size]
        decoder_tokens = np.zeros((len(article), TARGET_SEQ_LEN - 1), dtype="int32")
        decoder_tokens[:, 0] = vocab_y.index("[CLS]")
        generated_ids = [[] for _ in range(len(article))]
        done = np.zeros(len(article), dtype=bool)

        for position in range(TARGET_SEQ_LEN - 1):
            y_pred = model.predict(
                [article, decoder_tokens], batch_size=len(article), verbose=0
            )
            next_ids = np.argmax(y_pred[:, position], axis=-1)

            for i, next_id in enumerate(next_ids):
                next_id = int(next_id)
                if done[i] or next_id == end_id or next_id == pad_id:
                    done[i] = True
                    continue

                generated_ids[i].append(next_id)

                if position + 1 < TARGET_SEQ_LEN - 1:
                    decoder_tokens[i, position + 1] = next_id

            if done.all():
                break

        for ids in generated_ids:
            result = tokenizer_y.detokenize(tf.constant(ids, dtype=tf.int32))
            generated_summary = np.asarray(result).item()
            if isinstance(generated_summary, bytes):
                generated_summary = generated_summary.decode("utf-8")
            summaries.append(generated_summary)

    return summaries

In [5]:
decoder_input_train = y_train_pad[:, :-1]
decoder_target_train = y_train_pad[:, 1:]

decoder_input_val = y_validation_pad[:, :-1]
decoder_target_val = y_validation_pad[:, 1:]

model.evaluate(
    [X_validation_pad, decoder_input_val],
    decoder_target_val,
    sample_weight=(decoder_target_val != 0).astype("float32"),
    return_dict=True
)

313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - acc: 0.3128 - loss: 1.0375


{'acc': 0.3128341734409332, 'loss': 1.0375126600265503}

In [ ]:
#overall evaluation
decoder_input_test = y_test_pad[:, :-1]
decoder_target_test = y_test_pad[:, 1:]

predict = model.predict([X_test_pad, decoder_input_test],
                       batch_size=1,
                       verbose=1)

KeyboardInterrupt: 

In [21]:
import json
from tqdm import tqdm

candidates = indenpendent_generate_summary(X_test_pad, batch_size=64)
references = [
    r.decode("utf-8") if isinstance(r, bytes) else r
    for r in y_test
]

assert len(candidates) == len(references)

with open("validation_headlines_full.json", "w", encoding="utf-8") as f:
    json.dump({"candidates": candidates, "references": references}, f)

print("Saved pairs:", len(candidates))

Generating batches: 100%|██████████| 31/31 [03:44<00:00,  7.23s/it]

Saved pairs: 1951


In [22]:
import json

candidates = [x.decode("utf-8") if isinstance(x, bytes) else x for x in candidates]
references = [x.decode("utf-8") if isinstance(x, bytes) else x for x in references]

assert len(candidates) == len(references) > 0

with open("evaluation_backup.json", "x", encoding="utf-8") as f:
    json.dump({"candidates": candidates, "references": references}, f)

print("Saved pairs:", len(candidates))

Saved pairs: 1951


In [24]:
#bert score and rouge score
from bert_score import score as bert_score
from rouge_score import rouge_scorer

def as_text(items, name):
    result = []
    for i, item in enumerate(items):
        if isinstance(item, bytes):
            item = item.decode("utf-8")
        if not isinstance(item, str):
            raise TypeError(f"{name}[{i}] is {type(item).__name__}, not text")
        result.append(item)
    return result

candidates = as_text(candidates, "candidates")
references = as_text(references, "references")

if not candidates or len(candidates) != len(references):
    raise ValueError("Candidates and references must have equal, nonzero lengths")

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
rouge = [
    scorer.score(ref, cand)["rougeL"].fmeasure
    for ref, cand in zip(references, candidates)
]

P, R, F1 = bert_score(
    candidates, references,
    lang="en",
    rescale_with_baseline=True,
    device="cuda",
    batch_size=1,
)

print("Mean ROUGE-L F1:", sum(rouge) / len(rouge))
print("Mean BERTScore F1:", F1.mean().item())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Mean ROUGE-L F1: 0.18312565061672453
Mean BERTScore F1: 0.02277970314025879
